# Healthcare Patient Deterioration Analytics
**Student:** Vansh Jain \
**Technologies Used:** Azur Data Factory · Databricks PySpark · Unity Catalog · Spark MLlib \
**Notebook:** Gold Layer\
**Layers:** Bronze → Silver → Gold

## Import Libraries

In [0]:
from pyspark.sql.functions import(
    col,
    when,
    count,
    avg,
    round,
    sum,
    expr
)

### Storage Account Configuration

In [0]:
storage_account = 'healthcaremedalliondev'
storage_key = '<Your Storage Key>'

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

In [0]:
silver_path = f'abfss://silver@{storage_account}.dfs.core.windows.net/silver_patient_vitals'
gold_path = f'abfss://gold@{storage_account}.dfs.core.windows.net/'

### Read silver table

In [0]:
silver_df = (
    spark.read
    .format('delta')
    .load(silver_path)
)

In [0]:
print(f'Silver Record : {silver_df.count()}')
silver_df.printSchema()
display(silver_df.limit(5))

## Creating Multiple tables for Power BI Data Analysis

## Gold table 1 - Patient Summary

This table is designed for the Executive Dashboard.

KPI	                     |   Description
-------------------------|-----------------------------
total_patients	         |   Total patient observations
avg_heart_rate	         |   Average Heart Rate
avg_respiratory_rate	 |   Average Respiratory Rate
avg_spo2	             |   Average Oxygen Saturation
avg_temperature	Average  |   Body Temperature
avg_systolic_bp	Average  |   Systolic BP
avg_diastolic_bp	     |   Average Diastolic BP
avg_pulse_pressure	     |   Average Pulse Pressure
avg_sepsis_risk	Average  |   Sepsis Risk Score
deterioration_rate	     |   % of deteriorated patients

In [0]:
gold_patient_summary = silver_df.agg(
    count('*').alias('total_patients'),
    round(avg('heart_rate'), 2).alias('avg_heart_rate'),
    round(avg('respiratory_rate'), 2).alias('avg_respiratory_rate'),
    round(avg('spo2_pct'), 2).alias('avg_spo2'),
    round(avg('temperature_c'), 2).alias('avg_temperature'),
    round(avg('systolic_bp'), 2).alias('avg_systolic_bp'),
    round(avg('diastolic_bp'), 2).alias('avg_diastolic_bp'),
    round(avg('pulse_pressure'), 2).alias('avg_pulse_pressure'),
    round(avg('sepsis_risk_score'), 2).alias('avg_sepsis_risk'),
    round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
)

In [0]:
# validation

display(gold_patient_summary)

In [0]:
(
    gold_patient_summary.write
    .format('delta')
    .mode('overwrite')
    .save(gold_path + 'gold_patient_summary')
)

In [0]:
patient_summary = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_patient_summary')
)
display(patient_summary)

## Gold Table 2 - Hourly Vitals

Column	             |   Aggregation
---------------------|------
hour_from_admission	 |  Group By
avg_heart_rate	     |   Average
avg_respiratory_rate |	Average
avg_spo2	         |   Average
avg_temperature	     |  Average
avg_systolic_bp	     |   Average
avg_diastolic_bp	 |   Average
avg_pulse_pressure	 |   Average
avg_sepsis_risk	     |   Average
deterioration_rate	 |   Percentage

In [0]:
gold_hourly_vitals = (
    silver_df
    .groupby('hour_from_admission')
    .agg(
        count("*").alias("patient_observations"),
        round(avg('heart_rate'),2).alias('avg_heart_rate'),
        round(avg('respiratory_rate'),2).alias('avg_respiratory_rate'),
        round(avg('spo2_pct'),2).alias('avg_spo2'),
        round(avg('temperature_c'),2).alias('avg_temperature'),
        round(avg('systolic_bp'),2).alias('avg_systolic_bp'),
        round(avg('diastolic_bp'),2).alias('avg_diastolic_bp'),
        round(avg('pulse_pressure'),2).alias('avg_pulse_pressure'),
        round(avg('sepsis_risk_score'),2).alias('avg_sepsis_risk'),
        round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
    )
    .orderBy('hour_from_admission')
)

In [0]:
display(gold_hourly_vitals)

In [0]:
(
    gold_hourly_vitals.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + 'gold_hourly_vitals')
)

In [0]:
hourly_table = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_hourly_vitals')
)

display(hourly_table)

## Gold Table 3 - Admission Analysis

Analyze whether Emergency Department (ED), Elective, or Transfer admissions exhibit different clinical characteristics.

Column	             |   Description
---------------------|------------------------
admission_type	     |  ED / Elective / Transfer
patient_observations |	Number of records
avg_age	             |  Average age
avg_heart_rate	     |  Average HR
avg_spo2	         |  Average SpO₂
avg_temperature	     |  Average temperature
avg_sepsis_risk	     |  Average sepsis risk
deterioration_rate	 |  Percentage of deteriorated patients

In [0]:
gold_admission_analysis = (
    silver_df
    .groupby('admission_type')
    .agg(
        count('*').alias('patient_observations'),
        round(avg('age'),0).alias('avg_age'),
        round(avg('heart_rate'), 2).alias('avg_heart_rate'),
        round(avg('spo2_pct'), 2).alias('avg_spo2'),
        round(avg('temperature_c'), 2).alias('avg_temperature'),
        round(avg('sepsis_risk_score'), 2).alias('avg_sepsis_risk'),
        round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
    )
    .orderBy('admission_type')
)

In [0]:
display(gold_admission_analysis)

In [0]:
(
    gold_admission_analysis.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + 'gold_admission_analysis')
)

In [0]:
admission_table = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_admission_analysis')
)
display(admission_table)

## Gold Table 4 - Age Group Analysis

We will use four age groups:

Age	     |   Age Group
---------|---------
0 - 20   |   Young
21 - 40  |   Adult
41 - 60  |   Middle Age
61+	     |   Senior

In [0]:
age_df = silver_df.withColumn(
    'age_group',
    when(col('age') <= 20, "Young")
    .when((col('age') > 20) & (col('age') <= 40), 'Adult')
    .when((col('age') > 40) & (col('age') <= 60), 'Middle Age')
    .otherwise('Senior')
)

In [0]:
gold_age_group_analysis = (
    age_df
    .groupBy('age_group')
    .agg(
        count('*').alias('patient_observations'),
        round(avg('age'),0).alias('avg_age'),
        round(avg('heart_rate'), 2).alias('avg_heart_rate'),
        round(avg('spo2_pct'), 2).alias('avg_spo2'),
        round(avg('temperature_c'), 2).alias('avg_temperature'),
        round(avg('sepsis_risk_score'), 2).alias('avg_sepsis_risk'),
        round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
    )
)

In [0]:
gold_age_group_analysis = (
    gold_age_group_analysis
    .withColumn(
        'sort_order',
        expr("""
             case
                when age_group = 'Young' then 1
                when age_group = 'Adult' then 2
                when age_group = 'Middle Age' then 3
                when age_group = 'Senior' then 4
             end
             """)
    )
    .orderBy('sort_order')
    .drop('sort_order')
)

In [0]:
display(gold_age_group_analysis)

In [0]:
(
    gold_age_group_analysis.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + "gold_age_group_analysis")
)

In [0]:
age_group_table = (
    spark.read
    .format('delta')
    .load(gold_path + "gold_age_group_analysis")
)
display(age_group_table)

## Gold Table 5 - Sepsis Risk Analysis
Group By - high_sepsis_risk

Column	               | Description
-----------------------|--------------
high_sepsis_risk	   | High / Low Risk
patient_observations   | Number of observations
avg_sepsis_score	   | Average sepsis score
avg_lactate	           | Average lactate level
avg_wbc	               | Average white blood cell count
avg_crp	               | Average CRP level
avg_temperature	       | Average temperature
deterioration_rate	   | Percentage of deteriorated patients

In [0]:
gold_sepsis_analysis = (
    silver_df
    .groupBy('high_sepsis_risk')
    .agg(
        count('*').alias('patient_observations'),
        round(avg('sepsis_risk_score'),2).alias('avg_sepsis_score'),
        round(avg('lactate'), 2).alias('avg_lactate'),
        round(avg('wbc_count'), 2).alias('avg_wbc'),
        round(avg('crp_level'), 2).alias('avg_crp'),
        round(avg('temperature_c'), 2).alias('avg_temperature'),
        round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
    )
    .orderBy('high_sepsis_risk')
)

In [0]:
display(gold_sepsis_analysis)

In [0]:
(
    gold_sepsis_analysis.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + 'gold_sepsis_analysis')
)

In [0]:
sepsis_table = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_sepsis_analysis')
)
display(sepsis_table)

## Gold Table 6 - Oxygen Device Analysis

Group by = 'oxygen_device'

Column	                |Description
------------------------|--------------------
oxygen_device	        |Oxygen delivery method
patient_observations	|Number of observations
avg_oxygen_flow	        |Average oxygen flow
avg_spo2	            |Average oxygen saturation
avg_sepsis_risk	        |Average sepsis risk score
avg_temperature	        |Average body temperature
deterioration_rate	    |Percentage of deteriorated patients

In [0]:
gold_oxygen_analysis = (
    silver_df
    .groupBy('oxygen_device')
    .agg(
        count('*').alias('patient_observations'),
        round(avg('oxygen_flow'), 2).alias('avg_oxygen_flow'),
        round(avg('spo2_pct'), 2).alias('avg_spo2'),
        round(avg('sepsis_risk_score'), 2).alias('avg_sepsis_score'),
        round(avg('temperature_c'), 2).alias('avg_temperature'),
        round(avg('deterioration_next_12h') * 100, 2).alias('deterioration_rate')
    )
    .orderBy('oxygen_device')
)

In [0]:
display(gold_oxygen_analysis)

In [0]:
(
    gold_oxygen_analysis.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + 'gold_oxygen_analysis')
)

In [0]:
oxygen_table = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_oxygen_analysis')
)
display(oxygen_table)

## Gold Table 7 - High Risk Patients

high_sepsis_risk == 1

Category	    |Columns
----------------|----------
Time	        |hour_from_admission
Vitals	        |heart_rate, respiratory_rate, spo2_pct, temperature_c, systolic_bp, diastolic_bp, pulse_pressure
Laboratory	    |wbc_count, lactate, creatinine, crp_level, hemoglobin
Risk	        |sepsis_risk_score, high_sepsis_risk, deterioration_next_12h
Demographics	    |age, gender, admission_type
Oxygen	            |oxygen_device, oxygen_flow
Engineered Flags	|is_fever, is_tachycardia, is_hypoxic

In [0]:
gold_high_risk_patients = (
    silver_df
    .filter(col("high_sepsis_risk") == 1)
    .select(
        "hour_from_admission",
        "age",
        "gender",
        "admission_type",
        "heart_rate",
        "respiratory_rate",
        "spo2_pct",
        "temperature_c",
        "systolic_bp",
        "diastolic_bp",
        "pulse_pressure",
        "oxygen_device",
        "oxygen_flow",
        "wbc_count",
        "lactate",
        "creatinine",
        "crp_level",
        "hemoglobin",
        "sepsis_risk_score",
        "high_sepsis_risk",
        "is_fever",
        "is_tachycardia",
        "is_hypoxic",
        "deterioration_next_12h"
    )
)

In [0]:
display(gold_high_risk_patients)

In [0]:
(
    gold_high_risk_patients.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(gold_path + 'gold_high_risk_patients')
)

In [0]:
high_risk_table = (
    spark.read
    .format('delta')
    .load(gold_path + 'gold_high_risk_patients')
)

print(high_risk_table.count())

display(high_risk_table.limit(10))